Before you turn this problem in, make sure everything runs as expected. First, **restart the kernel** (in the menubar, select Kernel$\rightarrow$Restart) and then **run all cells** (in the menubar, select Cell$\rightarrow$Run All).

Make sure you fill in any place that says `YOUR CODE HERE` or "YOUR ANSWER HERE", as well as your name and collaborators below:

###### Version 2026.1

In [23]:
NAME = "Samantha A. Salazar"

# Presenting Uncertainty
## School of Information, University of Michigan

## Week 2: Assignment Overview
### The objectives for this week are for you to:

- review the concept of standard error
- construct a confidence distribution
- use a confidence distribution to construct a density plot, interval plot, CCDF barplot, and quantile dotplot
- apply one or more of the above techniques to the World Happiness dataset

In [24]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import altair as alt
import random
from scipy import stats
from scipy.stats import norm
from mads.lib.path import assets

# Part 1. Standard error and confidence distributions (14 points)

## 1.1 Bootstrap sampling distribution of phone pickups example

Recall in assignment 1 we used a sample from a hypothetical dataset of counts of how many times students at a school picked up their phones. Let's generate such a hypothetical dataset again. We'll make a sample of 20 students and the number of times each student picked up their phone:

In [25]:
np.random.seed(1234)   # for reproducibility

pickups_df = pd.DataFrame({"pickups": np.random.poisson(lam=40, size=20)})

pickups_df.describe()

,pickups
count,20.000000
mean,39.350000
std,6.072154
min,27.000000
25%,36.750000
50%,38.500000
75%,43.250000
max,52.000000


### Question 1.1.1 (2 points)

Using what you learned in assignment 1, generate a bootstrap sampling distribution of the mean number of pickups and assign it to the variable `bootstrap_means`:

In [26]:
B = 5000
np.random.seed(1234) # for reproducibility

# resample the observed pickups w/ replacement B times
# for each bootstrap sample, save the sample mean

bootstrap_means = np.array([pickups_df["pickups"].sample(n = len(pickups_df), replace = True).mean()
    for _ in range(B)
])

In [27]:
# the bootstrap sampling mean should be close to the sample mean
sample_mean = pickups_df['pickups'].mean()
sample_mean_se = np.std(pickups_df['pickups'], ddof=1) / np.sqrt(len(pickups_df))
assert len(bootstrap_means) == 5000, "Bootstrap sampling distribution: length of bootstrap_means should be 5000"
assert np.abs(np.std(bootstrap_means) - sample_mean_se) / sample_mean_se < 0.05, "Bootstrap sampling distribution: SD of boostrap_means does not match SE of sample_pickups"
assert np.abs(np.mean(bootstrap_means) - sample_mean)/sample_mean < 5*1e-3, "Bootstrap sampling distribution: mean of bootstrap_means does not match sample_mean"

### Question 1.1.2 (2 points)

Visualize the bootstrap sampling distribution of the mean as a density plot.

Hint: see assignment 1.

In [28]:
# put the bootstrap means in a dataframe so Altair can estimate/plot the density

bootstrap_df = pd.DataFrame({"Bootstrap sampling distribution of mean pickups": bootstrap_means})

pickups_bootstrap_chart = alt.Chart(bootstrap_df).transform_density(
    "Bootstrap sampling distribution of mean pickups",
    as_ = ["Bootstrap sampling distribution of mean pickups", "density"]
).mark_line(color = "gray").encode(
    x = "Bootstrap sampling distribution of mean pickups:Q",
    y = "density:Q"
)

pickups_bootstrap_chart

alt.Chart(...)

## 1.2 Standard error and Normal sampling distribution for mean pickups

The sampling distribution of a mean approaches the shape of a Normal distribution as the sample size increases. Instead of using bootstrapping to derive the distribution, we could instead use the *standard error*, which is an estimate of the standard deviation of the sampling distribution, assuming the sampling distribution is Normal.

The standard error of the mean ($s_\bar{x}$) is calculated using the sample standard deviation ($s$) and the sample size ($n$):

$s_\bar{x} = \frac{s}{\sqrt{n}}$

This calculation can be done as follows:

In [29]:
# manual calculation of standard error
pickups_se = pickups_df['pickups'].std(ddof=1) / np.sqrt(len(pickups_df))
print("Manual calculation of SE:           ", pickups_se)

# or using scipy.stats.sem() (should be the same)
pickups_se = stats.sem(pickups_df['pickups'])
print("scipy.stats.sem() calculation of SE:", pickups_se)


Manual calculation of SE:            1.357774882511437
scipy.stats.sem() calculation of SE: 1.357774882511437


Given the sample mean pickups ($\bar{x}$ = `pickups_mean`) and the sample mean standard error ($s_\bar{x}$ = `pickups_se`), we can construct a sampling distribution using the standard error. As discussed in lecture, there are three functions we'll want to define in order to construct different uncertainty visualizations: the probability density function (PDF), the cumulative distribution function (CDF), and the quantile function (also called the percentage points function, PPF). For a mean and standard error, these are:

$\begin{align}
\mathrm{density~function~(PDF):~} & f_\mathrm{Normal}\left(x \middle| \bar{x}, s_\bar{x}\right) & \mathtt{stats.norm.pdf()}\\
\mathrm{cumulative~distribution~function~(CDF):~} & F_\mathrm{Normal}\left(x \middle| \bar{x}, s_\bar{x}\right) & \mathtt{stats.norm.cdf()}\\
\mathrm{quantile~function~(PPF):~} & F^{-1}_\mathrm{Normal}\left(p \middle| \bar{x}, s_\bar{x}\right) & \mathtt{stats.norm.ppf()}
\end{align}$

In [30]:
pickups_mean = pickups_df['pickups'].mean()

# generate evenly-spaced numbers covering the range of values of pickups
# that we will calculate the density of the sampling distribution at
x = np.linspace(
    start = pickups_df['pickups'].min(), 
    stop = pickups_df['pickups'].max(),
    num = 1001
)

# this is the density of the sampling distribution: f_Normal(x | x_bar, se)
density = stats.norm.pdf(x, pickups_mean, pickups_se)

df = pd.DataFrame({"Normal sampling distribution of mean pickups": x, "density": density})

pickups_se_chart = alt.Chart(df).mark_line().encode(
    x="Normal sampling distribution of mean pickups",
    y="density"
)

pickups_se_chart

alt.Chart(...)

## 1.3 t-based confidence distribution for mean pickups

A slight improvement to the sampling distribution above is to use a Student's t confidence distribution. At small sample sizes, the Normal approximation is less accurate; a scaled-and-shifted Student's t distribution will have slightly fatter tails than the Normal distribution. 

To determine how fat the tails are, the t distribution uses a *degrees of freedom* parameter, which for the estimate of a single independent mean of sample size $n$ is equal to $n - 1$. Thus, an improved confidence distribution over the Normal approximation above is:

$\begin{align}
\mathrm{density~function~(PDF):~} & f_\mathrm{t}\left(x \middle| n - 1, \bar{x}, s_\bar{x}\right) & \mathtt{stats.t.pdf()}\\
\mathrm{cumulative~distribution~function~(CDF):~} & F_\mathrm{t}\left(x \middle| n - 1, \bar{x}, s_\bar{x}\right) & \mathtt{stats.t.cdf()}\\
\mathrm{quantile~function~(PPF):~} & F^{-1}_\mathrm{t}\left(p \middle| n - 1, \bar{x}, s_\bar{x}\right) & \mathtt{stats.t.ppf()}
\end{align}$

This confidence distribution is related to the common Student's t test: confidence intervals from this distribution are the same as the confidence intervals you would generate from a t test.

### Question 1.3.1 Visualize the Student's t confidence distribution for the mean (5 points)

Visualize the density of the t-based confidence distribution described above in a similar manner to how the Normal sampling distribution was visualized, but use a different color for the line (this will help you with the next question):


In [31]:
# use the t confidence distribution with n - 1 degrees of freedom
dof = len(pickups_df) - 1
x = np.linspace(
    start = pickups_df["pickups"].min(),
    stop = pickups_df["pickups"].max(),
    num = 1001
)

t_density = stats.t.pdf(x, df = dof, loc = pickups_mean, scale = pickups_se)

t_df = pd.DataFrame({
    "t confidence distribution of mean pickups": x,
    "density": t_density
})

pickups_t_chart = alt.Chart(t_df).mark_line(color = "red").encode(
    x = "t confidence distribution of mean pickups:Q",
    y = "density:Q"
)

pickups_t_chart

alt.Chart(...)

### Question 1.3.2 Visualize and compare uncertainty distributions (5 points)

Visualize all three distributions you have constructed so far in a single plot. 

Hint: if you have saved your previous plots in three separate variables and made them with different enough encodings that they can be distinguished by eye, this answer can be as simple as `pickups_bootstrap_chart + pickups_se_chart + pickups_t_chart`

In [ ]:
# overlay the bootstrap, Normal, and t-based uncertainty distributions
pickups_bootstrap_chart + pickups_se_chart + pickups_t_chart

Compare these distributions: what do you notice? Write your answer below.

---

### ANSWER:

The bootstrap, normal, and t-based distributions each have an estimated average number of pickups is centered around 39–40 pickups. The normal and t distributions look smoother because they are theoretical curves based on formulas, while the bootstrap distribution looks a little bumpier because it is created by repeatedly resampling the actual data. Even with that small difference, the overall shape and spread are very close across all three graphs. This suggests that the uncertainty around the mean is fairly consistent no matter which method is used, and most likely average pickup values fall in a narrow range around the center rather than being spread very far out.

# Part 2. Intervals, CDFs, and quantile dotplots (17 points)

Now that we are able to define the density, CDF, and quantile functions to describe our uncertainty in a mean, we can use these to construct various uncertainty visualizations as described in the lectures. For example, we can use the density function to create density plots or gradients; the CDF to create complementary CDF bar plots; and the quantile function to create intervals and quantile dotplots.

## 2.1 Intervals

### Question 2.1.1 Calculate 95% interval (2 points)

Using the quantile function (aka percentage points function) of the Student's t confidence distribution you derived in Part 1, calculate the lower and upper bounds of a 95% confidence interval for mean pickups and assign these bounds to the `pickups_lower` and `pickups_upper` variables. 

(Hint: use `stats.t.ppf()` as the quantile function and see the lecture on intervals.)

In [ ]:
# a 95% interval leaves 2.5% probability in each tail
dof = len(pickups_df) - 1
pickups_lower = stats.t.ppf(0.025, df=dof, loc=pickups_mean, scale=pickups_se)
pickups_upper = stats.t.ppf(0.975, df=dof, loc=pickups_mean, scale=pickups_se)

pickups_lower, pickups_upper

In [ ]:
dof = len(pickups_df) - 1
m = pickups_df['pickups'].mean()
se = stats.sem(pickups_df['pickups'])
#hiddent tests below

### Question 2.1.2 Visualize 95% + 66.666% interval (5 points)

Visualize a combined 95% + 66.666% (i.e. 2/3rds) interval for the mean of pickups, overlaid on top of the data (as ticks). 

Hints: 
- create two different layers, one for the 95% interval and one for the 66.666% interval, then combine them using `alt.layer()` or `+`.
- use `mark_rule(size=XXX).encode(x=YYY, x2=ZZZ)` to create each interval, where you fill in the values of `size` (to set the thickness of the interval), `x` (to set the lower bound), and `x2` (to set the upper bound).

Your output should look like this:

![Plot of 66% and 95% confidence intervals overliad on data](assignment2_intervals.png)

In [ ]:
# use the t distribution to calculate two uncertainty intervals around the sample mean
# the degrees of freedom are based on the sample size minus 1
dof = len(pickups_df) - 1

# the 95% interval shows the wider range of plausible mean pickup values
interval_95 = stats.t.ppf(
    [0.025, 0.975],
    df = dof,
    loc = pickups_mean,
    scale = pickups_se
)
# the 66.666% interval shows the narrower, more central range of plausible values
interval_66 = stats.t.ppf(
    [1/6, 5/6],
    df = dof,
    loc = pickups_mean,
    scale = pickups_se
)

# set a shared x-axis domain so the raw data ticks and interval bars align correctly
x_axis = alt.X(
    "lower:Q",
    scale = alt.Scale(domain = [0, 55])
)

# plot each observed pickup value as a red tick mark
# the y value controls vertical placement; smaller values move marks upward
data_ticks = alt.Chart(pickups_df).mark_tick(
    size = 18,
    thickness = 1,
    color = "red"
).encode(
    x = alt.X(
        "pickups:Q",
        scale = alt.Scale(domain = [0, 55]),
        title = "lower"
    ),
    y = alt.value(12)
)

# plot the wider 95% confidence interval as a thinner black bar
# x gives the lower bound and x2 gives the upper bound
ci_95 = alt.Chart(pd.DataFrame({
    "lower": [interval_95[0]],
    "upper": [interval_95[1]]
})).mark_rule(
    size = 4,
    color = "black"
).encode(
    x = alt.X("lower:Q", scale = alt.Scale(domain = [0, 55]), title = "lower"),
    x2 = "upper:Q",
    y = alt.value(11)
)

# plot the narrower 66.666% confidence interval as a thicker black bar
# using the same y value centers it on top of the 95% interval
ci_66 = alt.Chart(pd.DataFrame({
    "lower": [interval_66[0]],
    "upper": [interval_66[1]]
})).mark_rule(
    size = 8,
    color = "black"
).encode(
    x = alt.X("lower:Q", scale=alt.Scale(domain=[0, 55]), title="lower"),
    x2 = "upper:Q",
    y = alt.value(11)
)

# layer the raw observations and both interval bars into one compact uncertainty chart
# width controls horizontal stretch; height controls the amount of vertical space
alt.layer(
    data_ticks,
    ci_95,
    ci_66
).properties(
    width = 400,
    height = 23
)

## 2.2 Cumulative distribution function

### Question 2.2.1 Visualize a Complementary CDF barplot (5 points)

Using the CDF function (or the CCDF function), plot a CCDF barplot for the Student's t-based confidence distribution. *The barplot should start at 0*.

Hint: start from the code for generating a density plot, above, and (1) adjust the code for generating `x` to ensure it includes 0 and (2) change the function from a PDF to 1 - the CDF.

Your output should look something like this:

![CCDF barplot of mean pickups](assignment2_ccdf.png)

In [ ]:
# create x values starting at 0 so the CCDF barplot begins at 0
x = np.linspace(0, 55, 500)

# calculate the complementary CDF for the t-based confidence distribution
ccdf = 1 - stats.t.cdf(
    x,
    df = dof,
    loc = pickups_mean,
    scale = pickups_se
)

# put the x values and CCDF values into a dataframe for Altair
pickups_ccdf_df = pd.DataFrame({
    "x": x,
    "ccdf": ccdf
})

# plot the complementary CDF as a bar chart
pickups_ccdf_chart = alt.Chart(pickups_ccdf_df).mark_bar().encode(
    x = alt.X(
        "x:Q",
        title = "Mean pickups (CCDF)",
        scale = alt.Scale(domain = [5, 50]),
        axis = alt.Axis(
            labelFontSize = 10,        # x-axis number values
            titleFontSize = 10,        # x-axis title
            titleFontWeight = "bold"   # bold x-axis title
        )
    ),

    y=alt.Y(
        "ccdf:Q",
        title = "ccdf",
        scale = alt.Scale(domain=[0, 1]),
        axis = alt.Axis(
            values = [0.0, 0.5, 1.0],
            labelFontSize = 10,        # y-axis number values
            titleFontSize = 10,        # y-axis title
            titleFontWeight = "bold"   # bold x-axis title
        )
    )
).properties(
    width = 400,
    height = 43
)

pickups_ccdf_chart

## 2.3 Quantile dotplots

Constructing quantile dotplots involves first generating a small-to-medium number of quantiles from the quantile function (aka percentage points function) of the uncertainty distribution. We do this by projecting evenly-spaced values in probability space back into the data units through the quantile function, then stacking up those values into a dotplot:

<img src="cdf.png" alt="two graphs stacked on top of each other. The top graph has a y-axis of p less than x, from 0.00 to 1.00 in intervals of 0.25, and the x-axis is x, from 0 to 30. The cumulative distribution function line starts out around (0, 0.00) and stays that way until about (8, 0.00) where it rises up the y axis until it reaches about (18, 1.00) and continues on to 30. The bottom graph has a y-axis of count, from 0.00 to 1.00 in intervals of 0.25, and the x-axis is x, from 0 to 30. Circles are stacked between 8 and 18 on the x-axis, with a peak around (11, 0.50) and is slightly skewed to the right." style="width: 500px;"/>

The first step is to generate the evenly-spaced probabilities. We will create a `ppoints(n)` function that generates `n` evenly-spaced probabilities (not including 0 and 1):

In [ ]:
def ppoints(n):
    return (np.arange(n) + 0.5)/n

ppoints(20)

We can then use `ppoints` with the quantile function of the t confidence distribution (`stats.t.ppf`) to construct a quantile dotplot.

Dotplots in Altair are complex to construct, requiring three transformations:

1. `transform_bin` to bin the values
2. `transform_window` to rank the values within each bin (this stacks up all the dots in one bin)
3. `transform_joinaggregate` to calculate the midpoint of each bin using the median of all values in that bin

Putting together `ppoints`, the quantile function of the confidence distribution, and the above steps, we can construct a quantile dotplot.

In [ ]:
df = pd.DataFrame({
    "x": stats.t.ppf(ppoints(50), len(pickups_df) - 1, pickups_mean, pickups_se)
})

alt.Chart(df).transform_bin(
    field="x",
    as_="bin",
    bin=alt.BinParams(step=0.75)
).transform_window(
    rank_in_bin="rank()",
    groupby=["bin"]
).transform_joinaggregate(
    bin_midpoint="median(x)",
    groupby=["bin"]
).mark_circle(size=90).encode(
    x="bin_midpoint:Q",
    y="rank_in_bin:Q"
).properties(width=600, height=100)

### Question 2.3.1 Construct a 20-dot quantile dotplot for the mean of pickups (5 points)

Construct a 20-dot quantile dotplot for the mean of pickups. Adjust the binning step size, circle size, and/or chart size to achieve a closely-packed quantile dotplot like the 50-dot dotplot above. It is typically also necessary to adjust the step size in the binning (`alt.BinParams(step=XXX)`), the size of the circles (`mark_circle(size=XXX)`) and/or the width and height of the plot (`properties(width=XXX, height=YYY)`) to achieve good-looking dotplots.

In [ ]:
# generate 20 evenly spaced probability points and convert them into
# possible mean pickup values using the t-distribution quantile function
# this creates the values that will become the dots in the quantile dotplot
df = pd.DataFrame({
    "x": stats.t.ppf(
        ppoints(20),
        len(pickups_df) - 1,
        pickups_mean,
        pickups_se
    )
})

# build a 20-dot quantile dotplot from the generated pickup values
alt.Chart(df).transform_bin(
    # group nearby x values into bins so dots with similar values stack together
    field = "x",
    as_= "bin",
    bin = alt.BinParams(step = 0.9)
).transform_window(
    # rank the dots within each bin to create the vertical stacking effect
    rank_in_bin = "rank()",
    groupby = ["bin"]
).transform_joinaggregate(
    # use the median value in each bin as the x-position for that stack of dots
    bin_midpoint = "median(x)",
    groupby = ["bin"]
).mark_circle(
    # control the visual size of each dot
    size = 150
).encode(
    x = alt.X(
        # place each stack at the midpoint of its bin
        "bin_midpoint:Q",
        title = "Mean pickups",
        scale = alt.Scale(domain = [34, 44]),
        axis = alt.Axis(
            labelFontSize = 10, 
            titleFontSize = 11, 
            titleFontWeight = "bold"
        )
    ),
    y = alt.Y(
        # show how many dots are stacked in each bin
        # higher values mean more quantiles fall in that pickup range
        "rank_in_bin:Q",
        title = "Dot Count w/in Each Bin",
        axis = alt.Axis(
            values = [1, 2, 3, 4, 5],
            labelFontSize = 10, 
            titleFontSize = 11, 
            titleFontWeight = "bold"
        )
    )
).properties(
    # control the overall chart size
    width = 430,
    height = 80
)

## Part 3. Visualize world happiness (10 points)

We will do some visualization of the World Happiness Report dataset from 2015. It is a survey of the state of global happiness, which ranks 155 countries by their happiness levels. Let's explore the happiness score.

First, we'll read in the data and gather the happiness score for all countries:

In [ ]:
#Read in 2015 World Happiness Report statistics as a dataframe
file = assets.find("2015.csv")
happ = pd.read_csv(file)

#show the top 15 countries:
happ.head(15)

In [ ]:
#some descriptive statistics of the World Happiness Report:
happ.describe()

## 3.1 Happiness Scores

We'll focus on two columns in particular: the `Happiness Score` (an aggregate happiness score for each country) and the `Standard Error`, which is the standard error of the `Happiness Score`: because the data are based on surveys, there is uncertainty in the measurement of the happiness score, and this is quantified using its standard error.

### Question 3.1.1 Visualize the happiness scores and uncertainty of the top 15 countries (5 points)

Use one of the uncertainty visualizations above to visualize the happiness score of each country. You will need to use the Normal approximation to the sampling distribution along with the standard error of the happiness score to do this.

You **MUST NOT** use just a combination of a point and a single interval to visualize the scores. For example, the following chart **IS NOT AN ACCEPTABLE SOLUTION** (but could be a good place to start):

![point estimates and 95% confidence intervals for the happiness score in the top 10 countries according to the World Happiness Report](assignment2_happiness_intervals.png)

But you could construct a chart with a similar layout, but which uses one of the uncertainty encodings you learned above. For example, you might start by constructing a chart with 95% intervals, then extend it to be a chart with 95%+66.666% intervals, which would be an acceptable solution.

Hints:

1. You may want to sort the y axis by happiness score using something like `alt.Y("Country", sort=["Happiness Score"]`
2. You may want to restrict the x axis domain to make it easier to see the uncertainty using something like `alt.X("Happiness Score", scale={"domain":[6,8]})`
3. If you decide to make density plots, quantile dotplots, or CCDF barplots, you need to use the `facet()` function to display multiple subcharts where each one is a single country. If you do that, you may want to set the `"labelAngle"` property on the facet headers so they are horizontal.

In [ ]:
# focus on the first 15 rows, which represent the top 15 countries by happiness score
# .copy() avoids changing the original happiness dataframe
top15 = happ.head(15).copy()

# calculate the lower bound of the 95% Normal-based uncertainty interval
# this uses the country's happiness score as the mean and its standard error as the spread
top15["lower_95"] = stats.norm.ppf(
    0.025,
    loc = top15["Happiness Score"],
    scale = top15["Standard Error"]
)

# calculate the upper bound of the 95% uncertainty interval
top15["upper_95"] = stats.norm.ppf(
    0.975,
    loc = top15["Happiness Score"],
    scale = top15["Standard Error"]
)

# alculate the lower bound of the narrower 66.666% uncertainty interval
# this represents the more central range of likely happiness scores
top15["lower_66"] = stats.norm.ppf(
    1/6,
    loc = top15["Happiness Score"],
    scale = top15["Standard Error"]
)

# calculate the upper bound of the 66.666% uncertainty interval
top15["upper_66"] = stats.norm.ppf(
    5/6,
    loc = top15["Happiness Score"],
    scale = top15["Standard Error"]
)

# create the y-axis for country names and sort countries from highest to lowest happiness score
country_axis = alt.Y(
    "Country:N",
    sort = alt.SortField(field="Happiness Score", order = "descending"),
    title = "Country",
    axis = alt.Axis(labelFontSize=10)
)

# use the same x-axis scale for all chart layers so the intervals and points align
x_scale = alt.Scale(domain=[6, 8])

# draw the wider 95% uncertainty interval as a thin light-gray line
# this shows the broader plausible range for each country's happiness score
interval_95 = alt.Chart(top15).mark_rule(
    size = 3,
    color = "lightgray"
).encode(
    x = alt.X(
        "lower_95:Q",
        title = "Happiness Score",
        scale = x_scale,
        axis = alt.Axis(
            labelFontSize=10,
            titleFontSize=11,
            titleFontWeight="bold"
        )
    ),
    x2="upper_95:Q",
    y=country_axis
)

# draw the narrower 66.666% uncertainty interval as a thicker black line
# this emphasizes the more central range of likely values
interval_66 = alt.Chart(top15).mark_rule(
    size = 10,
    color = "black"
).encode(
    x = alt.X("lower_66:Q", scale=x_scale),
    x2 = "upper_66:Q",
    y = country_axis
)

# ddd a point for each country's reported happiness score
# the blue point marks the main estimate, while the black outline helps it stand out
points = alt.Chart(top15).mark_point(
    filled=True,
    size=50,
    color = "steelblue",
    stroke = "black",
    strokeWidth = 0.5
).encode(
    x = alt.X("Happiness Score:Q", scale = x_scale),
    y = country_axis,
    tooltip = ["Country:N", "Happiness Score:Q", "Standard Error:Q"]
)

# layer the 95% interval, 66.666% interval, and point estimate into one chart
# the title and subtitle explain that the chart shows both scores and uncertainty
(interval_95 + interval_66 + points).properties(
    width = 650,
    height = 350,
    title = alt.TitleParams(
        text = "Top 15 World Happiness Scores",
        subtitle = "Point estimates with 95% and 66.666% uncertainty intervals",
        anchor = "middle",
        fontSize = 14,
        subtitleFontSize = 11
    )
)

### Question 3.1.2 Reflect on the chart you created (5 points)

What are the pros and cons of the uncertainty encoding you used (be sure to have at least 2 pros and 2 cons)? How might it compare to other options you could have chosen?

---

### ANSWER:

I used a nested interval plot where each country has a point estimate, a wider 95% and a lower 66.6% interval. One major advantage is that it is easy to compare countries because the chart keeps the same ranked layout as a regular bar or dot plot while also adding uncertainty information. Another plus is that the two interval widths commmunicate uncertainty more clearly than a single interval view. The thinner gray line shows the broader plausible range, while the thicker black line emphasizes the more central range of likely scores.

A limitation of the interval plot is that they can sometimes seem abstract because it does not illustrate the full shape of the distribution. Viewers can see the range of uncertainty, but not whether the uncertainty is symmetric, skewed, or concentrated in certain areas. An additional con is that when intervals overlap, it is difficult to interpret quickly. 

In relation to other options, this chart offers a compact and neat way to compare many countries at once, whilke density and CCDF plots would like likely require facetting or require more space. However, dot plots and CCDF plots are able to illustrate the distribution more intuitively, so there are tradeoffs for each chart. 

When you are finished with your assignment, be sure to click on the submit button in Vocareum (upper right corner of the user interface). Please remember to work on your explanations and interpretations.